In [8]:
# !pip install "unstructured[pdf]"
# !brew install poppler
# !brew install tesseract
# !pip install -qU langchain-unstructured
# !pip install langchain_qdrant
# from langchain_community.document_loaders import DirectoryLoader

In [9]:
import os
import glob
import logging
from py_modules.document_loader import DocumentLoader, UnstructuredPDFMethod
from pprint import pprint
import warnings
warnings.filterwarnings("ignore")

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_ollama import OllamaEmbeddings


from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.llms import OllamaLLM

from langchain import hub
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import fitz
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image


def plot_pdf_with_boxes(pdf_page, segments):
    pix = pdf_page.get_pixmap()
    pil_image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(pil_image)
    categories = set()
    category_to_color = {
        "Title": "orchid",
        "Image": "forestgreen",
        "Table": "tomato",
    }
    for segment in segments:
        points = segment["coordinates"]["points"]
        layout_width = segment["coordinates"]["layout_width"]
        layout_height = segment["coordinates"]["layout_height"]
        scaled_points = [
            (x * pix.width / layout_width, y * pix.height / layout_height)
            for x, y in points
        ]
        box_color = category_to_color.get(segment["category"], "deepskyblue")
        categories.add(segment["category"])
        rect = patches.Polygon(
            scaled_points, linewidth=1, edgecolor=box_color, facecolor="none"
        )
        ax.add_patch(rect)

    # Make legend
    legend_handles = [patches.Patch(color="deepskyblue", label="Text")]
    for category in ["Title", "Image", "Table"]:
        if category in categories:
            legend_handles.append(
                patches.Patch(color=category_to_color[category], label=category)
            )
    ax.axis("off")
    ax.legend(handles=legend_handles, loc="upper right")
    plt.tight_layout()
    plt.show()


def render_page(doc_list: list, page_number: int, print_text=True) -> None:
    pdf_page = fitz.open(file_paths[0]).load_page(page_number - 1)
    page_docs = [
        doc for doc in doc_list if doc.metadata.get("page_number") == page_number
    ]
    segments = [doc.metadata for doc in page_docs]
    plot_pdf_with_boxes(pdf_page, segments)
    if print_text:
        for doc in page_docs:
            print(f"{doc.page_content}\n")

## File Paths

In [10]:
# Creating file paths
DATA_DIR = "./DataFiles/PDFs"
file_paths = glob.glob(os.path.join(DATA_DIR, "*.pdf"))
print(*file_paths, sep="\n")

./DataFiles/PDFs/layout-parser-paper.pdf


## Document Loading

In [11]:
# Instantiate document loader and set parsing strategy
loader = DocumentLoader(parsing_method=UnstructuredPDFMethod())

# Read pdf files
documents = []
for file in file_paths:
    documents += loader.get_document(file_path=file)

pprint(f"Total {len(documents)} pages were read from {len(file_paths)} files.")
# pprint(documents[0])

INFO: Reading PDF for file: ./DataFiles/PDFs/layout-parser-paper.pdf ...


'Total 181 pages were read from 1 files.'


In [12]:
# render_page(documents, 4)

In [13]:
# # Splitting the documents into chunks
# splitter = RecursiveCharacterTextSplitter(
#                                         chunk_size=1000,
#                                         chunk_overlap=200,
#                                         add_start_index=True
#                                         )

# documents_split = splitter.split_documents(documents)

# print(f"Total {len(documents_split)} splits were created from {len(documents)} pages.")

In [14]:
# documents_cat = [x.metadata['category'] for x in documents]
# print(documents_cat)

# unique_categories = set(documents_cat)
# print(unique_categories)

# Image category documents
# image_docs = [x for x in documents if x.metadata['category'] == 'Image']
# image_docs[0].page_content

# Documents without category "Image"
documents_filtered = [x for x in documents if x.metadata['category'] != 'Image']
print(f"Total {len(documents_filtered)} documents without category 'Image'.")

Total 175 documents without category 'Image'.


## Vector Database

In [15]:
# Initializing the Embedding model and Vector Store
embedding = OllamaEmbeddings(model="llama3.2")

client = QdrantClient(":memory:")

client.create_collection(
                        collection_name="demo_collection",
                        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
                        )


vector_store = QdrantVectorStore(
                                client=client,
                                collection_name="demo_collection",
                                embedding=embedding,
                                )


# Add the documents to the Vector Store
vector_store.add_documents(documents_filtered) #changed from documents_split to documents

# Setting up the retrievar
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 6})

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


## Prompt

In [16]:
# Using a predefined prompt
# prompt = hub.pull("rlm/rag-prompt")

system_message = """
You are an assitant who helps in answering questions based on the context provided to you. Provide the paper title for upon which you base your response.
Answer with 'I do not know or I am not sure' if you cannot find answers in the provided context.
"""

human_message = """Based on the following context.
Context: {context}.
---
Answer the question : {question}.

In your answer qoute the provided content metadata at the end of the answer.
Answer : """

prompt = ChatPromptTemplate(
    [
        ("system", system_message),
        ("human", human_message)

    ]
)

In [17]:
# parser = StrOutputParser()

# chain = prompt 

# response = chain.invoke({"question" : "my question", "context" : "my provided context"})
# print(parser.parse(response).to_messages()[1].content)

In [18]:
# example_messages = prompt.invoke({
#                                 "context": "filler context", 
#                                 "question": "filler question"
#                                 }).to_messages()

# print(example_messages)

## LLM chain creation

In [20]:
model_name = "llama3.2"
llm = OllamaLLM(model=model_name)

In [21]:
# documents[0].metadata

In [22]:
# Function to combine the retrieved documents
def format_docs(docs):
    # print(len(docs))
    context_string = ""
    for doc in docs:
        context_string += f"""Content:- \n{doc.page_content}\n---\nMetadata:-\nFile name : {doc.metadata["filename"]} | Page number : {doc.metadata["page_number"]} | Category : {doc.metadata["category"]}\n---\n\n"""
    
    # print("="*100)
    # print(context_string)
    # print("="*100)
    
    return context_string
    # return "\n\n".join(f"{doc.page_content}" for doc in docs)


rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
            )

## QA Examples

In [23]:
logging.getLogger().setLevel(logging.CRITICAL)

In [24]:
for chunk in rag_chain.stream("Deep learning methods for homography estimation."):
    print(chunk, end="", flush=True)

I do not know.

The provided context does not mention anything about deep learning methods for homography estimation. The topic seems to be related to document layout analysis and processing, but there is no information available on this specific question in the provided text.

Provided content metadata:
File name : layout-parser-paper.pdf | Page number : [insert page numbers] | Category : [FigureCaption/NarrativeText/ListItem]

In [25]:
for chunk in rag_chain.stream("Which images are availble in the provided paper?"):
    print(chunk, end="", flush=True)

I do not know or I am not sure which specific images are available in the provided paper. The text only mentions "Fig. 2: The relationship between the three types of layout data structures." but does not provide any visual content.

Metadata:
File name : layout-parser-paper.pdf | Page number : 6 | Category : FigureCaption
---

In [32]:
for chunk in rag_chain.stream("The relationship between the three types of layout data structure"):
    print(chunk, end="", flush=True)

The three types of layout data structures are Coordinate, TextBlock, and Layout object. 

Coordinate supports three kinds of variation and is a list of all possible layout elements, including other Layout objects. A TextBlock consists of coordinate information and extra features like block text, types, and reading orders.

LayoutParser: A Unified Toolkit for DL-Based DIA
File name : layout-parser-paper.pdf | Page number : 11 | Category : ListItem

Table 1: Current layout detection models in the LayoutParser model zoo
File name : layout-parser-paper.pdf | Page number : 5 | Category : FigureCaption

3.2 Layout Data Structures
File name : layout-parser-paper.pdf | Page number : 6 | Category : Title

LayoutParser: A Unified Toolkit for DL-Based DIA
File name : layout-parser-paper.pdf | Page number : 11 | Category : NarrativeText

In [26]:
for chunk in rag_chain.stream("Which figures are discussed in the provided paper?"):
    print(chunk, end="", flush=True)

Based on the provided context, I do not know which specific figure is being referred to as "Fig. 2" but it is mentioned that Fig. 2 discusses "The relationship between the three types of layout data structures.".

Metadata:
File name : layout-parser-paper.pdf | Page number : 6 | Category : FigureCaption

In [27]:
for chunk in rag_chain.stream("List the key findings in conclusion."):
    print(chunk, end="", flush=True)

I do not know.

Content Metadata:
File name : layout-parser-paper.pdf | Page number : 15 | Category : ListItem

In [28]:
for chunk in rag_chain.stream("What are the mentioned approachs in the papers?"):
    print(chunk, end="", flush=True)

Based on the context provided, it appears that the paper discusses various approaches to layout analysis and document image processing. The exact approaches mentioned in the text are not explicitly stated, but some general ideas can be inferred.

The text mentions "multiple layout detection models" as a potential approach for achieving optimal accuracy, especially for documents with complicated structures. Additionally, the paper highlights the importance of developing light-weight pipelines that are easy to use and require minimal model training or resource utilization.

It also seems that the authors propose using existing resources, such as the LayoutParser library, to build these pipelines and extract high-quality structured document data. Furthermore, the text mentions the use of deep convolutional neural networks for document image classification and retrieval, which could be considered a specific approach in this field.

Metadata:
File name : layout-parser-paper.pdf | Page numbe

In [29]:
for chunk in rag_chain.stream("Explain in detail about 'The Core LayoutParser Library'"):
    print(chunk, end="", flush=True)

Based on the provided context, I do not have enough information to provide a detailed explanation of "The Core LayoutParser Library" as it is not explicitly mentioned in the text.

However, based on the context, it can be inferred that the core library of LayoutParser likely consists of the underlying components and APIs that support the various layout detection models and pipelines described in the paper. The library appears to provide a unified toolkit for deep learning-based document image analysis (DIA), enabling practitioners to build efficient and accurate pipelines for extracting structured data from historical documents.

The text mentions that the library supports the same set of transformation and operation APIs across different layout elements, including TextBlock, Layout objects, and other layouts. This suggests that the core library provides a flexible and consistent framework for working with layout data structures.

Additionally, the paper highlights the importance of bu

In [30]:
for chunk in rag_chain.stream("How can LayoutParser be used for document analysis?"):
    print(chunk, end="", flush=True)

LayoutParser can be used to develop pipelines for extracting high-quality structured data from documents, particularly those with complicated layouts. It can be utilized for various document analysis tasks such as generating structured data from historical documents, financial tables, and other types of complex layouts.

For instance, LayoutParser was used in a project to generate high-quality structured data from historical Japanese firm financial tables with complicated layouts. The pipeline applied two layout models to identify different levels of document structures and two customized OCR engines for optimized character recognition accuracy.

LayoutParser can be used by practitioners in academia and industry to build pipelines that are focused on precision, efficiency, and robustness, making it a valuable tool for extracting structured data from complex documents.

Metadata:
File name : layout-parser-paper.pdf | Page number : 11 | Category : ListItem
---

In [ ]:
for chunk in rag_chain.stream("Discuss the use cases of LayoutParser."):
    print(chunk, end="", flush=True)

Based on the provided context, it appears that LayoutParser is a tool designed for layout detection and data annotation in various documents, including historical ones. The paper highlights its focus on precision, efficiency, and robustness, making it suitable for handling complicated document structures.

The use cases of LayoutParser include:

1. **Extracting structured data from historical documents**: The digitization of historical documents can unlock valuable data, but obtaining a structured representation is often complicated due to scan noises, page wearing, and complex layout structures. LayoutParser helps address this challenge by providing efficient layout detection tools.
2. **Building lightweight pipelines for simple documents**: For relatively simple documents, LayoutParser offers light-weight pipelines that prioritize development ease, speed, and flexibility. This makes it an ideal choice for practitioners in academia and industry who need to quickly build and deploy pip